In [34]:
from openai import OpenAI
openai_client = OpenAI()

In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [37]:
question = "What is the exact percentage score required on CS50 problem sets to get a certificate?"
answer = llm(question)
print(answer)

For **CS50’s standard certificate of completion**, there is **no exact percentage score required on problem sets**.

What matters is typically:

- **Completing the course requirements**
- **Submitting the required problem sets**
- **Passing the final project**
- Sometimes, depending on the platform/course version, meeting a **minimum overall course score**

If you mean **CS50’s online certificate via edX**, the usual benchmark is **70% overall**, not a specific percentage on the problem sets alone.

If you want, I can tell you the exact certificate requirement for:
- **CS50x on edX**
- **Harvard College CS50**
- **CS50’s free certificate**
- **A specific year/version of the course**


In [38]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [39]:
# Rather than the commented part below this, which is what Alexy ran to get the json format of his FAQ for zoomcamp
# I run the script saved in 'project' folder on the cs50 course:  https://cs50.harvard.edu/x/syllabus/ to get it into json format.


# import requests

# docs_url = "https://datatalks.club/faq/json/courses.json"
# response = requests.get(docs_url)
# courses_raw = response.json()

# courses_raw

In [40]:
# import json
# import os
# from bs4 import BeautifulSoup
# from openai import OpenAI
# from pydantic import BaseModel, Field
# import requests

# # Initialize OpenAI client (requires OPENAI_API_KEY env variable set)
# client = OpenAI()


# # 1. Define the target JSON structure
# class FAQItem(BaseModel):
#     id: str = Field(description="Sequential ID, e.g., cs50_faq_001")
#     category: str = Field(
#         description="Specific category (e.g., Grading, Regret Clause, AI Policy, Project, Resubmission)"
#     )
#     question: str = Field(
#         description="A distinct, single-topic operational question"
#     )
#     answer: str = Field(
#         description="Exact rule, score, percentage, or policy details without omitting facts"
#     )


# class FAQList(BaseModel):
#     items: list[FAQItem]


# def extract_max_cs50_json():
#     url = "https://cs50.harvard.edu/x/2026/syllabus/"
#     print(f"1. Fetching syllabus page from {url}...")

#     response = requests.get(url)
#     response.raise_for_status()

#     # 2. Clean HTML clutter
#     soup = BeautifulSoup(response.text, "html.parser")
#     for clutter in soup(["script", "style", "nav", "footer", "header", "aside"]):
#         clutter.extract()

#     clean_text = soup.get_text(separator="\n", strip=True)

#     print(
#         "2. Extracting maximum atomic FAQ items via OpenAI (no collapsing allowed)..."
#     )

#     # 3. Request maximum density extraction using strict system instructions
#     completion = client.beta.chat.completions.parse(
#         model="gpt-4o-mini",
#         messages=[
#             {
#                 "role": "system",
#                 "content": (
#                     "You are an exhaustive data extraction assistant. Parse the provided CS50 syllabus "
#                     "and extract EVERY SINGLE rule, policy, requirement, threshold, and restriction "
#                     "as its own separate, atomic FAQ item.\n\n"
#                     "CRITICAL EXTRACTION RULES:\n"
#                     "1. DO NOT summarize or combine rules into broad umbrella topics.\n"
#                     "2. Extract as many distinct items as possible (target 15 to 25 items).\n"
#                     "3. Separate distinct policies into individual items:\n"
#                     "   - Minimum score cutoff (70%) as one item.\n"
#                     "   - What happens if you score below 70% as a separate item.\n"
#                     "   - Resubmission mechanics as one item.\n"
#                     "   - Regret Clause deadline window (72 hours) as one item.\n"
#                     "   - Regret Clause consequences as a separate item.\n"
#                     "   - Allowed use of duck.cs50.ai as one item.\n"
#                     "   - Ban on third-party commercial AI (ChatGPT, Copilot, Claude) as a separate item.\n"
#                     "   - Unreasonable collaboration vs reasonable collaboration as separate items.\n"
#                     "   - Final project implementation rules as one item.\n"
#                     "   - Final project video presentation rules as a separate item.\n"
#                     "   - edX Certificate vs Harvard Division of Continuing Education certificate as separate items."
#                 ),
#             },
#             {"role": "user", "content": clean_text[:25000]},
#         ],
#         response_format=FAQList,
#     )

#     # 4. Save extracted items
#     parsed_items = completion.choices[0].message.parsed.model_dump()["items"]
#     output_filename = "cs50_faq.json"

#     with open(output_filename, "w", encoding="utf-8") as f:
#         json.dump(parsed_items, f, indent=2, ensure_ascii=False)

#     print(
#         f"3. Success! Extracted {len(parsed_items)} granular FAQ items to {output_filename}"
#     )
#     print(f"File location: {os.path.abspath(output_filename)}")


# if __name__ == "__main__":
#     extract_max_cs50_json()

In [41]:
# Sample of json data above extracted:
import json

# 1. Load the local JSON file directly
with open("cs50_faq.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

# 2. Check how many documents were loaded
len(documents)


18

In [42]:
# Print the top 2 items formatted nicely
print(json.dumps(documents[:2], indent=2))

[
  {
    "id": "cs50_faq_001",
    "category": "Certificates",
    "question": "What is the minimum score required to receive a verified certificate from edX?",
    "answer": "You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course\u2019s ten problem sets as well as on the course\u2019s final project."
  },
  {
    "id": "cs50_faq_002",
    "category": "Certificates",
    "question": "What happens if I score below 70% on a problem set or final project?",
    "answer": "If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate."
  }
]


In [43]:
# now that we have the json data let's set up the search and indexing for it, i.e. the R in RAG

# There are many search libraries we can use like Elasticsearch Solr but they are heavy duty search libraries;
# require things like docker. In our case since we have quite a small dataset, we will use minsearch - which is 
# a lightweight version of Elasticsearch and was created by Alexey Grigorev; I'll link the repo below. 

# So we import minsearch and; we create an Index class which required two fields: text_field and keyword_fields;
# text_fields are all the fields you can use to perform search. In our case it is the 'question' field, 'section' field and 'answer' field. In BMLL terms it is the values of the fields.
# keyword_fields are the things you need AN EXACT MATCH FOR.
# because you may use it in a database query like: SELECT * FROM index WHERE course = 'data-engineering-zoomcamp' ; so you need the exact string to match

# Explain also keyword_fields can be used to narrow the search down, open up my json and show, I have a few Categories.

from minsearch import Index

index = Index(
    text_fields=["question", "answer"],
    keyword_fields=["category"]
)

index.fit(documents)





In [44]:
question = 'What is the exact percentage score required on CS50 problem sets to get a certificate?' 
index.search(question)  

# index.search uses some text-matching under the hood, 
# it calculates a relevance score relative to the question you are asking vs the items in your json dictionaries.
# And it sorts it from highest to lowest.

# In our case I have 18 json dictionaries, and the search will likely return all 18, but if you had a million say, it would
# only return 500-1000 items.

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to receive a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate.'},
 {'id': 'cs50_faq_014',
  'category': 'Problem Sets',
  'question': 'How many problem sets do I need to submit?',
  'answer': 'You are expected to submit ten problem sets.'},
 {'id': 'cs50_faq_006',
  'category': 'AI Policy',
  'question': 'Is the use of duck.cs50.ai allowed?',
  'answer': 'Yes, the use of duck.cs50.ai is allowed.'},
 {'id': 'cs50_faq_018',
  'category': 'Problem Sets',
 

In [45]:
# A bit cheecky because I know the question has a direct answer in the webpage
# Check which category it is in: Certificates in this case, 
# in our case the HTML page is 1 page but if you had 10,000 pages of data that are sorted into different courses you could filter for the
# specific course you are after for example.

index.search(question, filter_dict={'category':'Certificates'})  # returns all the questions only from the Certificates section

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to receive a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate.'}]

In [46]:
# And you can also filter the number of results you want to see
search_results =index.search(question, filter_dict={'category':'Certificates'}, num_results=5) # if we just want 5 results

In [47]:
# Going back to our high level RAG function: 
# this was our original RAG function
# def rag(question):
#     search_results = search(question)
#     user_prompt = build_prompt(question, search_results)
#     return llm(user_prompt)


# We now have enough information to set up the search() function: 
# Last thing to cover is the boost keyword; you can put more weight on certain text_fields;
# So if your question had the word CS500 in it, and two different json dictionaries had CS500, one in the question only and the other in the 
# answer only the json dict with it in the question will be matched twice as high as the one with it in the question only. 
# Reminder it is relative to 1. 
def search(question, category='Certificates'):
    boost_dict={'question':2.0}
    filter_dict={'category':category}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [48]:
search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.
search_results

[{'id': 'cs50_faq_001',
  'category': 'Certificates',
  'question': 'What is the minimum score required to receive a verified certificate from edX?',
  'answer': 'You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.'},
 {'id': 'cs50_faq_002',
  'category': 'Certificates',
  'question': 'What happens if I score below 70% on a problem set or final project?',
  'answer': 'If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate.'}]

In [49]:
"""
We move on to the A in RAG, which is Augmented, but he calls it PROMPT.
"""

'\nWe move on to the A in RAG, which is Augmented, but he calls it PROMPT.\n'

In [50]:
# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [51]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [52]:
# Each document becomes a block with the section, question, and answer. 
# This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. 
# It's a small preprocessing step before we send the data to the LLM.
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["category"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [53]:
# Now we combine the question with the context into the user prompt:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()



prompt = build_prompt(question, search_results)

print(prompt)


# You should see a prompt with the question at the top and several FAQ entries below it. This is exactly what we'll send to the LLM.
# Which is exactly what we see in the output. 

# The prompt is the bridge between search and the LLM. 
# A bad prompt lets the LLM ignore the context and hallucinate. A good prompt keeps the answer grounded.

# Prompt engineering is part art, part science. You experiment, try different things, and see what works. 

Question:
What is the exact percentage score required on CS50 problem sets to get a certificate?

Context:
Certificates
Q: What is the minimum score required to receive a verified certificate from edX?
A: You must receive a satisfactory score (at least 70%) on each problem you submit as part of one of the course’s ten problem sets as well as on the course’s final project.

Certificates
Q: What happens if I score below 70% on a problem set or final project?
A: If you score below 70% on any problem set or the final project, you will not be eligible for a verified certificate.


In [54]:
"""
The last component in RAG is Generation, which is the llm. It takes the prompt we built and generates an answer.
"""

'\nThe last component in RAG is Generation, which is the llm. It takes the prompt we built and generates an answer.\n'

In [55]:
# Going back to the start where we prompted ChatGPT directly. 
# OpenAI has a response API or chat completion AI - legacy. Grok and Gemini

# If we feed the prompt from above into the model
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [56]:
# We can see the usage, or how much it costs us
# https://developers.openai.com/api/docs/models/gpt-5.4-mini


response.usage

ResponseUsage(input_tokens=135, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=29, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=164)

In [57]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

# So you can see it cost us a very small fraction of a penny to run the prompt. It is a very cheap model to run. 

0.00023175000000000002

In [58]:
# Before building the final prompt and putting everything together, one last thing we need is message history. 
# WHen you have a chat with ChatGPT, you will have noticed that the questions or prompts you give it after you have 
# a lengthy conversation with it, ChatGPT will use the conversation you have with it to answer your latest question. 

# We won't build a multi-turn chat here. But we still use this message format to separate our instructions from the user prompt.

message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [59]:
# We can now put this together into an updated llm function.
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [61]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


answer = rag("What is the exact percentage score required on CS50 problem sets to get a certificate?")
print(answer)

You must score **at least 70%** on each CS50 problem set to be eligible for a certificate.
